# Ethan Li
## CSCI E-89 Deep Learning, Assignment 03, Problem 5

This notebook trains, evaluates, visualizes, predicts with, and tunes a PyTorch Fashion-MNIST MLP. It is self-contained and is intended to run with **Restart & Run All**.

> Build a PyTorch image classifier for Fashion MNIST in a new folder prob5/ (ignore all other files in this repo). Do not run code or install packages; I will run everything locally. Just write the files carefully so they run the first time, and commit each script separately.
> 
> Scripts must also work when pasted into consecutive Jupyter cells (no argparse, plots shown inline, later cells reuse names from earlier cells if the .py files aren't importable). Add a .gitignore for the dataset and __pycache__/.
> 
> 1. data.py: download Fashion MNIST with torchvision, convert to normalized float32 tensors, seed 42, split the 60,000 training images into 55,000 train / 5,000 validation, DataLoaders with batch size 32 (only train shuffled).
> 2. model.py: MLP nn.Module: flatten 28x28, hidden layers of 300 and 100 with ReLU (configurable sizes), 10 output logits.
> 3. train.py: SGD (lr 0.1), cross-entropy, 20 epochs, torchmetrics multiclass accuracy (plain accuracy, average="micro"). Record and print mean train loss, train accuracy, and validation accuracy per epoch. Use a GPU if available. Then evaluate test accuracy, save the weights, and save the history (with test accuracy) as JSON.
> 4. plot.py: plot training and validation accuracy per epoch on one chart; save as PNG and show inline in Jupyter. train.py calls it after training.
> 5. predict.py: for the first 3 validation images, print predicted vs. true class names, softmax probabilities for all 10 classes (3 decimals), and the top 4 classes.
> 6. tune.py: seeded Optuna search over the learning rate (log scale 1e-5 to 1e-1), both hidden layer sizes, and SGD momentum; 10 trials of 5 epochs each. Print a table of trials and the best parameters, retrain the best configuration for 20 epochs, and compare its test accuracy with train.py's baseline.
> 
> Then: create PROMPTS.md, SUMMARY.md, and this unexecuted self-contained notebook; finish with exact local commands.

## `data.py`
Download Fashion-MNIST, normalize float32 image tensors, make the seeded split, and create loaders.

In [ ]:
import random
import numpy as np
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED, BATCH_SIZE, DATA_DIR = 42, 32, 'data'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2860,), (0.3530,))])
full_train_dataset = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=transform)
train_dataset, validation_dataset = random_split(full_train_dataset, [55000, 5000], generator=torch.Generator().manual_seed(SEED))
train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True)
validation_loader = DataLoader(validation_dataset, BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, BATCH_SIZE, shuffle=False)
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


## `model.py`
Define the configurable 28×28-image MLP with ReLU activations and ten output logits.

In [ ]:
import torch.nn as nn

class FashionMNISTMLP(nn.Module):
    # Hidden widths default to the required 300 and 100 units.
    def __init__(self, hidden_size1=300, hidden_size2=100):
        super().__init__()
        self.network = nn.Sequential(nn.Flatten(), nn.Linear(784, hidden_size1), nn.ReLU(), nn.Linear(hidden_size1, hidden_size2), nn.ReLU(), nn.Linear(hidden_size2, 10))
    def forward(self, images):
        return self.network(images)


## `plot.py`
Provide the plotting function used by training; `plt.show()` renders inline in Jupyter and the figure is saved as PNG.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

def plot_history(history, output_path=Path('artifacts/fashion_mnist_accuracy.png')):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    epochs = range(1, len(history['train_accuracy']) + 1)
    figure, axis = plt.subplots(figsize=(8, 5))
    axis.plot(epochs, history['train_accuracy'], label='Training accuracy')
    axis.plot(epochs, history['validation_accuracy'], label='Validation accuracy')
    axis.set(xlabel='Epoch', ylabel='Accuracy', title='Fashion-MNIST MLP Accuracy', ylim=(0, 1))
    axis.grid(True, alpha=0.3); axis.legend(); figure.tight_layout()
    figure.savefig(output_path, dpi=150); plt.show()


## `train.py`
Train the baseline with SGD and micro accuracy, save weights and JSON history, test it, and call the plotting function.

In [ ]:
import json
from torch.optim import SGD
from torchmetrics.classification import MulticlassAccuracy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def score(model, loader, metric):
    model.eval(); metric.reset()
    with torch.no_grad():
        for images, labels in loader: metric.update(model(images.to(device)), labels.to(device))
    return metric.compute().item()

model = FashionMNISTMLP().to(device); optimizer = SGD(model.parameters(), lr=0.1); criterion = nn.CrossEntropyLoss()
train_metric = MulticlassAccuracy(num_classes=10, average='micro').to(device)
validation_metric = MulticlassAccuracy(num_classes=10, average='micro').to(device)
history = {'train_loss': [], 'train_accuracy': [], 'validation_accuracy': []}
for epoch in range(1, 21):
    model.train(); train_metric.reset(); loss_sum = 0.0; count = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device); optimizer.zero_grad()
        logits = model(images); loss = criterion(logits, labels); loss.backward(); optimizer.step()
        loss_sum += loss.item() * labels.size(0); count += labels.size(0); train_metric.update(logits, labels)
    mean_loss = loss_sum / count; train_accuracy = train_metric.compute().item(); validation_accuracy = score(model, validation_loader, validation_metric)
    history['train_loss'].append(mean_loss); history['train_accuracy'].append(train_accuracy); history['validation_accuracy'].append(validation_accuracy)
    print(f'Epoch {epoch:02d}/20: loss={mean_loss:.4f}, train_accuracy={train_accuracy:.4f}, validation_accuracy={validation_accuracy:.4f}')
test_accuracy = score(model, test_loader, MulticlassAccuracy(num_classes=10, average='micro').to(device)); history['test_accuracy'] = test_accuracy
print(f'Test accuracy: {test_accuracy:.4f}')
Path('artifacts').mkdir(exist_ok=True); torch.save(model.state_dict(), 'artifacts/fashion_mnist_mlp.pt')
with open('artifacts/fashion_mnist_history.json', 'w', encoding='utf-8') as file: json.dump(history, file, indent=2)
plot_history(history)


## `predict.py`
Use the trained baseline to print predictions, every class probability, and the top four classes for three validation images.

In [ ]:
model.eval(); images, labels = next(iter(validation_loader))
with torch.no_grad(): probabilities = torch.softmax(model(images[:3].to(device)), dim=1).cpu()
for index, (probability, label) in enumerate(zip(probabilities, labels[:3]), 1):
    top_scores, top_labels = probability.topk(4)
    print(f'Image {index}: predicted={class_names[probability.argmax().item()]}, true={class_names[label.item()]}')
    print('  Probabilities: ' + ', '.join(f'{name}={value:.3f}' for name, value in zip(class_names, probability.tolist())))
    print('  Top 4: ' + ', '.join(f'{class_names[item.item()]} ({value.item():.3f})' for value, item in zip(top_scores, top_labels)))


## `tune.py`
Run a seeded Optuna search for ten five-epoch trials, print the table and best parameters, then retrain the winner for twenty epochs and compare test accuracy.

In [ ]:
import optuna

def objective(trial):
    torch.manual_seed(SEED + trial.number)
    lr = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True); h1 = trial.suggest_int('hidden_size1', 100, 500, step=50); h2 = trial.suggest_int('hidden_size2', 50, 300, step=50); momentum = trial.suggest_float('momentum', 0.0, 0.9)
    candidate = FashionMNISTMLP(h1, h2).to(device); optimizer = SGD(candidate.parameters(), lr=lr, momentum=momentum); criterion = nn.CrossEntropyLoss()
    for _ in range(5):
        candidate.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device); optimizer.zero_grad(); loss = criterion(candidate(images), labels); loss.backward(); optimizer.step()
    return score(candidate, validation_loader, MulticlassAccuracy(num_classes=10, average='micro').to(device))

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED)); study.optimize(objective, n_trials=10)
print('number  validation_accuracy  parameters')
for trial in study.trials: print(f'{trial.number:>6}  {trial.value:>19.4f}  {trial.params}')
print('Best parameters:', study.best_params)
# Retrain the best configuration for 20 epochs before evaluating it on test data.
params = study.best_params; tuned_model = FashionMNISTMLP(params['hidden_size1'], params['hidden_size2']).to(device); optimizer = SGD(tuned_model.parameters(), lr=params['learning_rate'], momentum=params['momentum']); criterion = nn.CrossEntropyLoss()
for epoch in range(20):
    tuned_model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device); optimizer.zero_grad(); loss = criterion(tuned_model(images), labels); loss.backward(); optimizer.step()
tuned_test_accuracy = score(tuned_model, test_loader, MulticlassAccuracy(num_classes=10, average='micro').to(device))
print(f'Baseline test accuracy: {test_accuracy:.4f}'); print(f'Tuned test accuracy:    {tuned_test_accuracy:.4f}')
